# BONUS: Advanced Streaming Patterns

Advanced incremental ingestion patterns: Stream-Static Joins, Streaming Aggregations with Watermarking, `badRecordsPath` error handling, and Change Data Feed for Incremental ETL.

> **Note:** This notebook extends [05 — Incremental Ingestion](05_incremental_ingestion.ipynb). Run that notebook first to create required Delta tables and checkpoint directories.

In [0]:
%run ../../setup/00_setup

### Configuration

Import libraries and define paths for source data, checkpoints, and schema locations.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import time

## Stream-Static Joins & Aggregations

> **Databricks official definition — Structured Streaming:** "Structured Streaming is a scalable and fault-tolerant stream processing engine built on the Spark SQL engine. In Structured Streaming, a live data stream is treated as a table that is being continuously appended. You express streaming computations as standard batch-like queries on static tables, and Spark runs them incrementally and continuously."
> — *[Apache Spark / Databricks Documentation](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html)*

Stream-Static Join = enriching a stream (e.g., Orders) with a static table (e.g., Customers). Static table is re-read at each micro-batch start.

---

In [0]:
# Load Static Table
df_static_customers = spark.table(TABLE_CUSTOMERS)
display(df_static_customers)

### Demo: Stream-Static Join (Append Mode)

In [0]:
spark.sql("DROP TABLE IF EXISTS jointed_orders")

In [0]:
df_enriched.createOrReplaceTempView("enriched_orders_stream")
df_static_customers.createOrReplaceTempView("static_customers")

df_joined = spark.sql("""
SELECT
  o.order_id,
  o.total_amount,
  c.first_name,
  c.last_name,
  c.email,
  o._processing_time
FROM enriched_orders_stream o
LEFT JOIN static_customers c
  ON o.customer_id = c.customer_id
""")

In [0]:
# Write enriched stream
query_join = (df_joined.writeStream
    .format("memory")
    .queryName("enriched_orders")
    .outputMode("append") # Joins (Left Stream-Static) are append-only
    .option("checkpointLocation", f"{CHECKPOINT_BASE_PATH}/join_demo_3")
    .trigger(processingTime="5 seconds")
    .start()
)

In [0]:
# Simulate arrival of remaining batches (2 to 10)
print("Starting data simulation...")

for i in range(11, 13): # Batches 2 to 10 (indices of remaining parts)
    batch_num = i + 1
    print(f"Arriving: Batch {batch_num}...", end=" ")
    
    # Write next batch
    stream_batches[i].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_{batch_num:02d}")
    
    print("Done. Waiting for stream...")
    time.sleep(4) # Wait for trigger to pick it up

print("All batches arrived.")
display(spark.table("enriched_orders"))

In [0]:
query_join.stop()
display(spark.sql("SELECT count(1) FROM enriched_orders "))

### Demo: Streaming Aggregation (Watermarking)

In [0]:
# [1/2] Define windowed aggregation — count orders per customer per 30-second window
windowed_counts = (df_enriched
    .withWatermark("_processing_time", "1 minutes")  # allow 1 min late data
    .groupBy(
        F.window("_processing_time", "30 seconds"),
        "customer_id"
    )
    .count()
)
print("Streaming aggregation defined: count orders per (customer, 30s window)")

In [ ]:
# [2/2] Write stream — UPDATE mode: only emit changed windows (efficient for aggregations)
query_agg = (windowed_counts.writeStream
    .format("console")
    .queryName("orders_counts")
    .outputMode("update")
    .option("checkpointLocation", f"{CHECKPOINT_BASE_PATH}/agg_demo")
    .trigger(processingTime="5 seconds")
    .start()
)
print("Aggregation stream started...")

In [0]:
# Simulate arrival of remaining batches (2 to 10)
print("Starting data simulation...")

for i in range(14, 16): # Batches 2 to 10 (indices of remaining parts)
    batch_num = i + 1
    print(f"Arriving: Batch {batch_num}...", end=" ")
    
    # Write next batch
    stream_batches[i].write.mode("overwrite").json(f"{STREAM_SOURCE_PATH}/batch_{batch_num:02d}")
    
    print("Done. Waiting for stream...")
    time.sleep(4) # Wait for trigger to pick it up

print("All batches arrived.")
display(windowed_counts)

In [0]:
# Stop for demo purposes
query_agg.stop()

### Demo: badRecordsPath

In [0]:
TABLE_ERRORS = f"{BRONZE_SCHEMA}.customers_with_validation"

**Creating table with `_corrupt_record`:**

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE_ERRORS}")

spark.sql(f"""
CREATE TABLE {TABLE_ERRORS} (
  customer_id STRING,
  first_name STRING,
  last_name STRING,
  email STRING,
  phone STRING,
  city STRING,
  state STRING,
  country STRING,
  registration_date DATE,
  customer_segment STRING,
  _corrupt_record STRING,
  _ingestion_timestamp TIMESTAMP
) USING DELTA
""")

**Loading with error handling:**

In [0]:
# Create a CSV with a bad row
bad_csv_data = [
    (999, "Eve", "2023-01-03"),
    (888, "Frank", "NOT_A_DATE") # This will fail date parsing
]
spark.createDataFrame(bad_csv_data, ["customer_id", "first_name", "registration_date"]).write.mode("overwrite").option("header", "true").csv(f"{BATCH_SOURCE_PATH}/bad_csv")

In [0]:
df_with_errors = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("badRecordsPath", BAD_RECORDS_PATH)
    .schema("""
        customer_id STRING,
        first_name STRING,
        registration_date DATE,
        _corrupt_record STRING
    """)
    .load(f"{BATCH_SOURCE_PATH}/bad_csv")
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)
display(df_with_errors)

**Bad records statistics:**

In [0]:
files = [f.path for f in dbutils.fs.ls(BAD_RECORDS_PATH)]
files = [f + "*" for f in files]
print(files)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

bad_record_schema = StructType([
    StructField("path", StringType(), True),
    StructField("record", StringType(), True),
    StructField("reason", StringType(), True)
])

df_bad_records = (
    spark.read
    .format("json")
    .schema(bad_record_schema)
    .load(files)
)
display(df_bad_records)

## Change Data Feed for Incremental ETL

> **Databricks official definition — Change Data Feed (CDF):** "Change Data Feed allows Databricks to track row-level changes between versions of a Delta table. When enabled, the runtime records insert, update, and delete operations in the `_change_type` metadata column. You can read the change stream using `table_changes('table_name', version)` for efficient incremental ETL without reprocessing entire tables."
> — *[Databricks Documentation](https://docs.databricks.com/en/delta/delta-change-data-feed.html)*

In M03 we learned how to **enable CDF** and **read change data** with `_change_type` metadata. Now let's see how CDF enables **efficient incremental ETL pipelines**.

Instead of reprocessing entire tables, use CDF to process **only the rows that changed** since the last pipeline run.

---

### Example: Incremental ETL with CDF

**Objective:** Build an incremental pipeline that reads only changed rows from a CDF-enabled source table

**Pattern:**
1. Track the `last_processed_version` (checkpoint)
2. Read changes since that version using `readChangeFeed`
3. Filter for `insert` and `update_postimage` (skip pre-images and deletes)
4. Apply transformations and write to the target table

In [ ]:
# [1/2] Ensure CDF demo table exists — creates it if module M03 was skipped
if not spark.catalog.tableExists(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo"):
    spark.sql(f"""
    CREATE TABLE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo (
        user_id STRING, name STRING, email STRING,
        status STRING, updated_at TIMESTAMP
    ) USING DELTA
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)
    spark.sql(f"""
    INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.cdf_demo VALUES
        ('U001', 'Alice', 'alice@example.com', 'premium', current_timestamp()),
        ('U003', 'Charlie', 'charlie@example.com', 'active', current_timestamp()),
        ('U004', 'Diana', 'diana@example.com', 'trial', current_timestamp())
    """)
    print("CDF demo table created for this module")
else:
    print(f"Using existing cdf_demo table from M03")

In [ ]:
# [2/2] Display current table state — this is the baseline CDF will track changes from
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo"))

In [ ]:
# Simulate new changes to the source table
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.cdf_demo VALUES
    ('U005', 'Eve', 'eve@example.com', 'active', current_timestamp())
""")
spark.sql(f"""
UPDATE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo
SET status = 'active', updated_at = current_timestamp()
WHERE user_id = 'U004'
""")
print("New changes applied: INSERT U005, UPDATE U004 (trial → active)")

In [ ]:
# [1/2] Read CDF — all changes since a specific version (in production: read from checkpoint)
last_processed_version = 0

incremental_changes = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", last_processed_version) \
    .table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo")

print(f"Reading changes from version {last_processed_version} — all _change_type values:")
display(incremental_changes.select("user_id", "name", "_change_type", "_commit_version"))

In [ ]:
# [2/2] Filter + transform — keep only current-state records (skip pre-images and deletes)
transformed = incremental_changes \
    .filter(F.col("_change_type").isin(["insert", "update_postimage"])) \
    .withColumn("processed_at", F.current_timestamp()) \
    .withColumn("email_domain", F.split(F.col("email"), "@")[1])

print(f"Incremental processing — only relevant changes (from version {last_processed_version}):")
display(transformed.select(
    "user_id", "name", "email_domain", "status",
    "_change_type", "_commit_version", "processed_at"
))

**When to use each incremental method:**

| Method | Source | Best For |
|--------|--------|----------|
| **COPY INTO** | Files in cloud storage | Batch file ingestion with idempotency |
| **Auto Loader** | Files in cloud storage | Streaming file ingestion with schema evolution |
| **CDF** | Delta tables | Processing row-level changes between Delta tables |

> **Exam Note:** CDF is a **Delta-to-Delta** incremental method. For file-based ingestion, use COPY INTO or Auto Loader.

← [05 — Incremental Ingestion](05_incremental_ingestion.ipynb) | **[ README](../../../README.md)**